# Unit 5, Lecture 2: Validation and catching hallucinations

Last lecture tested that **your code** is correct. But your code can be perfect
and the agent can still fail, because **the model can be confidently wrong**. It
can invent a refund policy, cite a rule that does not exist, or return a number it
made up, and say it with total confidence.

Testing never catches this, because the bug is not in your code. You need a
**gate**: check the model's output against something you control before you act.
Three checks, cheapest to deepest:

1. **Structural**, is it the right shape and on the allowed menu?
2. **Grounding**, did the claim come from the source, or was it invented?
3. **Cross-check**, does a claimed fact match ground truth from a tool?

**The running example:** the real policy says **30 days**; the model confidently
says **60**. This whole notebook runs offline.

## 1. Structural: is it even on the menu?

The cheapest first gate. Reject off-menu choices and out-of-range numbers before
spending effort on deeper checks.

In [ ]:
from cse476.validation import validate_choice, validate_number

queues = {"billing", "technical", "account"}
print("on menu:  ", validate_choice("Billing.", queues))     # -> billing (cleaned)
print("off menu: ", validate_choice("the moon", queues))     # -> None (rejected)
print()
print("sane number:  ", validate_number("20%", 0, 100))       # -> 20.0
print("out of range: ", validate_number("150", 0, 100))       # -> None
print("not a number: ", validate_number("about ten", 0, 100)) # -> None

## 2. Grounding: is the claim actually in the source?

The heart of hallucination detection. If the agent answers from a document, every
fact must be findable in that document. The model kept the real words and swapped
the real number.

In [ ]:
from cse476.validation import is_grounded

policy = "Refunds are available within 30 days of purchase."

grounded = "refunds within 30 days"       # matches the source
invented = "refunds within 60 days"       # the model made up 60

print("'30 days' grounded?", is_grounded(grounded, policy))   # True
print("'60 days' grounded?", is_grounded(invented, policy))   # False -> HALLUCINATION

### Point at exactly what was invented

More useful than "something is wrong" is "the model invented **60**". Note this
catches invented **numbers** specifically, and numbers (prices, dates, dosages)
are the most common and most damaging hallucination.

In [ ]:
from cse476.validation import find_unsupported_claim

print("the invented token:", find_unsupported_claim("refunds within 60 days", policy))
print("a grounded claim:  ", find_unsupported_claim("within 30 days", policy))  # None

## 3. Cross-check: ask a tool that holds the truth

The strongest check. When you have a real source of truth, never let a confident
sentence override it.

In [ ]:
from cse476.validation import cross_check

# the model says the balance is 5000; the balance tool reads the real DB: 4200
model_says = 5000
tool_says  = 4200   # ground truth

print("model matches truth?", cross_check(model_says, tool_says))  # False
print("-> the tool wins. Reject the model's number, use 4200.")
print()
print("when they agree:", cross_check(4200, 4200))  # True

## The gate: turn a bad answer into a handled one

A validator is a gate between the model and the action. It returns a verdict, and
the agent acts only if valid, otherwise it falls back safely.

In [ ]:
from cse476.validation import validate_answer

policy = "Refunds are available within 30 days of purchase."

good = validate_answer("refunds within 30 days", policy)
bad  = validate_answer("refunds within 60 days", policy)

print("grounded answer:   ", good)
print("hallucinated answer:", bad)
print()

# how an agent uses the gate:
answer = "refunds within 60 days"
verdict = validate_answer(answer, policy)
if verdict["valid"]:
    print("ACT on:", answer)
else:
    print("FALL BACK, reason:", verdict["reason"])

## The mapping, and why validation matters

In [ ]:
from cse476.validation import VALIDATION_MAP, why_validate

for concept, meaning in VALIDATION_MAP.items():
    print(f"{concept:22} ->  {meaning}")
print()
for k, v in why_validate().items():
    print(f"{k:22}: {v}")

## Your turn

**1. Catch a hallucination.** Take a short source document and write a claim that
keeps its words but changes one number. Run `is_grounded` and confirm it catches
the invented number.

**2. Add a gate.** Put a `validate_answer` gate in front of one place your agent
uses a model answer. On an invalid verdict, fall back to a safe default instead of
acting.

**3. Find your source of truth.** For one claim your capstone makes, name the tool
or document that could verify it. If there is none, that claim is unguarded, and
you should know which of your claims are exposed.

In [ ]:
# your work here
